In [3]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import timm

# Hardware setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

# Set PLANTVILLAGE_DIR to a folder containing train/ and val/ directories.
data_dir = os.environ.get("PLANTVILLAGE_DIR")
if not data_dir:
    raise RuntimeError("Set PLANTVILLAGE_DIR before running this training notebook.")

# Increased to 384x384 to resolve fine lesion veins; batch size 64 fits easily in 16GB VRAM
IMG_SIZE = 384
BATCH_SIZE = 64
NUM_WORKERS = 6

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.25),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_transform)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS, 
    pin_memory=True, 
    persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=True, 
    persistent_workers=True
)

num_classes = len(train_dataset.classes)
print(f"Device: {device} | Classes: {num_classes}")
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

Device: cuda | Classes: 38
Train samples: 43444 | Val samples: 10861


In [2]:
# Modern backbone: ConvNeXt-Tiny pre-trained natively on 384x384 inputs
model = timm.create_model(
    'convnext_tiny.fb_in22k_ft_in1k_384', 
    pretrained=True, 
    num_classes=num_classes
).to(device)

# Label smoothing prevents overconfidence on ambiguous pairs
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# AdamW with weight decay handles higher capacity models cleanly
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

EPOCHS = 10
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

print("Model, optimizer, scaler, and scheduler ready.")

Model, optimizer, scaler, and scheduler ready.


In [3]:
best_macro_f1 = -1.0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    start_time = time.time()

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Automatic Mixed Precision (FP16 execution)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_dataset)

    # --- Validation Phase ---
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            preds = outputs.argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    val_macro_f1 = f1_score(all_labels, all_preds, average="macro")
    elapsed = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {train_loss:.4f} | Val F1: {val_macro_f1:.4f} | "
          f"LR: {current_lr:.2e} | Time: {elapsed:.1f}s")

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ↳ New best! Saved checkpoint (macro-F1={best_macro_f1:.4f})")

print(f"\nTraining Complete. Best Validation Macro-F1: {best_macro_f1:.4f}")

Epoch 01/10 | Loss: 0.8458 | Val F1: 0.9890 | LR: 9.76e-05 | Time: 367.5s
  ↳ New best! Saved checkpoint (macro-F1=0.9890)


Epoch 02/10 | Loss: 0.6975 | Val F1: 0.9845 | LR: 9.05e-05 | Time: 289.5s


Epoch 03/10 | Loss: 0.6885 | Val F1: 0.9887 | LR: 7.96e-05 | Time: 288.7s


Epoch 04/10 | Loss: 0.6841 | Val F1: 0.9938 | LR: 6.58e-05 | Time: 289.1s
  ↳ New best! Saved checkpoint (macro-F1=0.9938)


Epoch 05/10 | Loss: 0.6785 | Val F1: 0.9951 | LR: 5.05e-05 | Time: 290.5s
  ↳ New best! Saved checkpoint (macro-F1=0.9951)


Epoch 06/10 | Loss: 0.6760 | Val F1: 0.9960 | LR: 3.52e-05 | Time: 290.6s
  ↳ New best! Saved checkpoint (macro-F1=0.9960)


Epoch 07/10 | Loss: 0.6737 | Val F1: 0.9953 | LR: 2.14e-05 | Time: 284.9s


Epoch 08/10 | Loss: 0.6728 | Val F1: 0.9963 | LR: 1.05e-05 | Time: 288.8s
  ↳ New best! Saved checkpoint (macro-F1=0.9963)


Epoch 09/10 | Loss: 0.6724 | Val F1: 0.9968 | LR: 3.42e-06 | Time: 281.7s
  ↳ New best! Saved checkpoint (macro-F1=0.9968)


Epoch 10/10 | Loss: 0.6720 | Val F1: 0.9969 | LR: 1.00e-06 | Time: 281.1s
  ↳ New best! Saved checkpoint (macro-F1=0.9969)

Training Complete. Best Validation Macro-F1: 0.9969


In [4]:
model.load_state_dict(best_model_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

cm = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=train_dataset.classes, digits=4)

print("=== Classification Report (v3) ===")
print(report)

# Per-class lowest performance check
from sklearn.metrics import f1_score as f1_per_class
per_class_f1 = f1_per_class(all_labels, all_preds, average=None)
weakest = sorted(zip(train_dataset.classes, per_class_f1), key=lambda x: x[1])[:5]

print("\nWeakest 5 Classes:")
for name, score in weakest:
    print(f"  {name}: F1 = {score:.4f}")

=== Classification Report (v3) ===
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab     1.0000    1.0000    1.0000       126
                                 Apple___Black_rot     1.0000    1.0000    1.0000       125
                          Apple___Cedar_apple_rust     1.0000    1.0000    1.0000        55
                                   Apple___healthy     1.0000    0.9939    0.9970       329
                               Blueberry___healthy     0.9934    1.0000    0.9967       300
          Cherry_(including_sour)___Powdery_mildew     1.0000    1.0000    1.0000       210
                 Cherry_(including_sour)___healthy     1.0000    0.9941    0.9971       170
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     0.9703    0.9515    0.9608       103
                       Corn_(maize)___Common_rust_     1.0000    0.9958    0.9979       239
               Corn_(maize)___Northern_Leaf_

In [5]:
import pickle

bundle = {
    "architecture": "convnext_tiny",
    "num_classes": num_classes,
    "class_names": train_dataset.classes,
    "state_dict": model.state_dict(),
    "img_size": IMG_SIZE,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
}

output_path = "model_v3.pkl"
with open(output_path, "wb") as f:
    pickle.dump(bundle, f)

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Saved {output_path} ({size_mb:.1f} MB)")

Saved model_v3.pkl (106.3 MB)